# Mô hình đối chứng Facebook Prophet — đo trên tập Test niêm phong

**Dự án:** Tốt nghiệp - Energy Forecasting - **Nhóm thực hiện:** The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

Notebook này dựng mô hình đối chứng Prophet để tính **Forecast Skill Score** cho mô hình
LightGBM chính thức của dự án.

### Vì sao phải viết lại notebook này

Bản trước đặt `TY_LE_TRAIN = 0.8` rồi **tự cắt 80/20 bên trong file audit**. Nghĩa là
Prophet được chấm điểm trên 20% cuối chuỗi của riêng nó, còn LightGBM được chấm trên
tập test niêm phong — **hai tập dòng khác nhau**. Ghép hai con số đó lại thành Skill
Score là sai phép so sánh, dù kết quả có đẹp đến đâu.

### Điều kiện so sánh công bằng trong bản này

| Điều kiện | Cách làm |
|---|---|
| Cùng dữ liệu học | Prophet học trên tập **Development** (train + val) — đúng phần LightGBM được học |
| Cùng mốc dự báo | Prophet dự báo tại đúng các mốc thời gian mục tiêu $T+h$ của tập test |
| Cùng tập chấm điểm | Lấy thẳng từ `prediction_audit.parquet` — cùng tử số, cùng mẫu số |
| Cùng phạm vi | Cả giá trị tại $T$ lẫn nhãn tại $T+h$ đều phải là số đo thật |

Prophet **chỉ** học từ lịch sử sản lượng và chu kỳ ngày/tuần của chính nó, không được
đưa bất kỳ đặc trưng thời tiết nào vào. Đó là bản chất của một baseline chuỗi thời gian
thuần túy — đối lập có chủ đích với LightGBM có đầy đủ đặc trưng thời tiết.

### Tính tái lập

Prophet dùng chế độ ước lượng MAP (không lấy mẫu MCMC) nên với cùng dữ liệu đầu vào,
kết quả lặp lại giống nhau giữa các lần chạy. Notebook này dùng cùng một logic với
`srcs/05_machine_learning/pipeline/actions/baseline_prophet_test_set.py`, nên số ra ở
đây trùng với số của pipeline.

## 2. Import thư viện và khai báo tham số

In [2]:
import json
import time
import warnings
import logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

import numpy as np
import pandas as pd
from prophet import Prophet

# ── Đường dẫn ──
DEV_PATH = '../../data/model/v3/05_selected/v3_development_selected.parquet'
AUDIT_PATH = '../../data/model/v3/07_final_test/prediction_audit.parquet'
TEST_PATH = '../../data/model/v3/05_selected/v3_test_selected.parquet'
OUTPUT_DIR = Path('../../data/model/v3/08_baseline_prophet_test')

# ── Tham số ──
HORIZONS = [1, 4]        # h1 = 15 phút, h4 = 60 phút
BUOC_PHUT = 15           # lưới thời gian 15 phút
MIN_DONG_MOI_SITE = 200  # dưới ngưỡng này thì Prophet không đủ dữ liệu để học

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Đã import thư viện và khai báo tham số.')
print(f'- Tập học Prophet : {DEV_PATH}')
print(f'- Tập chấm điểm   : {AUDIT_PATH}')
print(f'- Ghi kết quả ra  : {OUTPUT_DIR}')

Đã import thư viện và khai báo tham số.
- Tập học Prophet : ../../data/model/v3/05_selected/v3_development_selected.parquet
- Tập chấm điểm   : ../../data/model/v3/07_final_test/prediction_audit.parquet
- Ghi kết quả ra  : ../../data/model/v3/08_baseline_prophet_test


## 3. Đọc tập học (Development) — chỉ giữ dòng đo thật

Prophet chỉ được nhìn phần dữ liệu mà LightGBM cũng được nhìn. Các dòng do bước điền
khuyết ETL sinh ra bị loại, vì học trên số liệu bịa sẽ làm baseline méo theo hướng
không kiểm soát được.

In [3]:
df_dev = pd.read_parquet(
    DEV_PATH, columns=['site_id', 'timestamp', 'energy_generated_kwh', 'energy_source']
)
print(f'Tổng số dòng tập Development: {len(df_dev):,}')

df_dev = df_dev[df_dev['energy_source'] == 'measured']
df_dev = df_dev.rename(columns={'timestamp': 'ds', 'energy_generated_kwh': 'y'})
df_dev = df_dev[['site_id', 'ds', 'y']]

print(f'Số dòng đo thật dùng để học: {len(df_dev):,}')
print(f'Số trạm: {df_dev["site_id"].nunique()}')
display(df_dev.head(3))

Tổng số dòng tập Development: 2,273,970
Số dòng đo thật dùng để học: 951,854
Số trạm: 42


,site_id,ds,y
24,1,2020-01-01 06:15:00,0.135
25,1,2020-01-01 06:30:00,0.465
26,1,2020-01-01 06:45:00,1.039


## 4. Đọc tập chấm điểm và xác định phạm vi

Phạm vi lấy đúng điều kiện của chỉ số công bố, cộng thêm một điều kiện nữa: **nhãn tại
$T+h$ cũng phải là số đo thật**. Chấm một dự báo bằng một nhãn do ETL bịa ra thì con số
không còn đo năng lực dự báo nữa.

In [4]:
df_audit = pd.read_parquet(AUDIT_PATH)
df_goc = pd.read_parquet(TEST_PATH, columns=['site_id', 'timestamp', 'energy_source'])
df_goc = df_goc.rename(columns={'energy_source': 'src_goc'})

df_audit = df_audit.merge(df_goc, on=['site_id', 'timestamp'], how='left')
df_audit = df_audit.sort_values(['site_id', 'timestamp']).reset_index(drop=True)

mask_cham = {}
for h in HORIZONS:
    # Nguồn của NHÃN tại T+h: dịch cột nguồn lên h bước trong từng trạm
    src_nhan = df_audit.groupby('site_id')['src_goc'].shift(-h)
    m = (
        (df_audit['energy_source'] == 'measured')        # giá trị tại T là đo thật
        & (src_nhan == 'measured')                        # nhãn tại T+h cũng đo thật
        & df_audit['is_daylight'].fillna(False).astype(bool)
        & df_audit[f'y_true_h{h}'].notna()
        & df_audit[f'y_pred_h{h}'].notna()
    )
    mask_cham[h] = m
    df_audit.loc[m, f'ds_muc_tieu_h{h}'] = (
        df_audit.loc[m, 'timestamp'] + pd.Timedelta(minutes=BUOC_PHUT * h)
    )
    print(f'h{h}: {int(m.sum()):,} dòng được chấm điểm')

print(f'\nTổng số dòng trong audit: {len(df_audit):,}')
display(df_audit[['site_id', 'timestamp', 'energy_source', 'is_daylight']].head(3))

h1: 220,114 dòng được chấm điểm
h4: 202,908 dòng được chấm điểm

Tổng số dòng trong audit: 486,160


,site_id,timestamp,energy_source,is_daylight
0,1,2021-12-18 09:30:00,measured,True
1,1,2021-12-18 09:45:00,measured,True
2,1,2021-12-18 10:00:00,measured,True


## 5. Hàm huấn luyện Prophet cho 1 trạm

Prophet học xu hướng và chu kỳ ngày/tuần. Tắt `yearly_seasonality` vì tập dữ liệu chưa
đủ nhiều năm để ước lượng chu kỳ năm cho ra hồn. Dự báo bị chặn dưới ở 0 — sản lượng
điện mặt trời không thể âm.

In [5]:
def train_prophet_1_tram(dev_tram, moc_can_du_bao):
    """Học Prophet trên lịch sử 1 trạm rồi dự báo tại các mốc thời gian cần.

    Trả về Series: index = mốc thời gian, value = sản lượng dự báo (kWh).
    """
    if len(dev_tram) < MIN_DONG_MOI_SITE or len(moc_can_du_bao) == 0:
        return None

    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
    )
    model.fit(dev_tram[['ds', 'y']].sort_values('ds'))

    du_bao = model.predict(pd.DataFrame({'ds': moc_can_du_bao}))
    return pd.Series(du_bao['yhat'].clip(lower=0).to_numpy(), index=moc_can_du_bao)


def tinh_wape(y_that, y_bao):
    """WAPE = tổng |sai số| / tổng |thực tế|, đơn vị %."""
    mau = np.abs(y_that).sum()
    return float(np.abs(y_that - y_bao).sum() / mau * 100.0) if mau > 0 else np.nan


print('Đã định nghĩa hàm train_prophet_1_tram() và tinh_wape().')

Đã định nghĩa hàm train_prophet_1_tram() và tinh_wape().


## 6. Chạy Prophet cho toàn bộ 42 trạm

Mỗi trạm mất khoảng 10–15 giây, tổng khoảng 8–10 phút.

In [6]:
t0 = time.time()
sites = sorted(df_audit['site_id'].unique())
ket_qua_tram, site_loi = [], []

for i, s in enumerate(sites, 1):
    dev_tram = df_dev[df_dev['site_id'] == s]
    audit_tram = df_audit[df_audit['site_id'] == s]

    # Gom mọi mốc thời gian mục tiêu cần dự báo cho trạm này (cả h1 lẫn h4)
    moc = pd.DatetimeIndex(sorted({
        t for h in HORIZONS
        for t in audit_tram.loc[mask_cham[h][audit_tram.index], f'ds_muc_tieu_h{h}'].dropna()
    }))

    try:
        du_bao = train_prophet_1_tram(dev_tram, moc)
    except Exception as e:
        site_loi.append({'site_id': s, 'loi': str(e)[:150]})
        print(f'   [{i:>2}/{len(sites)}] trạm {s}: LỖI — {str(e)[:70]}')
        continue
    if du_bao is None:
        continue

    ghi = {'site_id': s, 'n_train': len(dev_tram)}
    for h in HORIZONS:
        idx = audit_tram.index[mask_cham[h][audit_tram.index]]
        if len(idx) == 0:
            continue
        y_that = df_audit.loc[idx, f'y_true_h{h}'].to_numpy(float)
        y_bao = du_bao.reindex(df_audit.loc[idx, f'ds_muc_tieu_h{h}']).to_numpy(float)
        df_audit.loc[idx, f'prophet_h{h}'] = y_bao
        ok = ~np.isnan(y_bao)
        ghi[f'n_test_h{h}'] = int(ok.sum())
        ghi[f'wape_prophet_h{h}'] = round(tinh_wape(y_that[ok], y_bao[ok]), 4)
        ghi[f'wape_model_h{h}'] = round(
            tinh_wape(y_that[ok], df_audit.loc[idx, f'y_pred_h{h}'].to_numpy(float)[ok]), 4)
    ket_qua_tram.append(ghi)

    mo_ta = '  '.join(
        f"h{h}: Prophet {ghi.get(f'wape_prophet_h{h}', np.nan):.2f}% / "
        f"model {ghi.get(f'wape_model_h{h}', np.nan):.2f}%" for h in HORIZONS)
    print(f'   [{i:>2}/{len(sites)}] trạm {s}: {mo_ta}   ({(time.time()-t0)/60:.1f} phút)')

df_ket_qua = pd.DataFrame(ket_qua_tram)
print(f'\nĐã chạy xong {len(df_ket_qua)}/{len(sites)} trạm trong {(time.time()-t0)/60:.1f} phút.')
if site_loi:
    print(f'Số trạm lỗi: {len(site_loi)} — {[x["site_id"] for x in site_loi]}')
display(df_ket_qua.head(5))

01:29:48 - cmdstanpy - INFO - Chain [1] start processing
01:29:55 - cmdstanpy - INFO - Chain [1] done processing


   [ 1/40] trạm 1: h1: Prophet 67.07% / model 13.26%  h4: Prophet 63.17% / model 16.76%   (0.1 phút)


01:29:56 - cmdstanpy - INFO - Chain [1] start processing
01:30:02 - cmdstanpy - INFO - Chain [1] done processing


   [ 2/40] trạm 2: h1: Prophet 65.28% / model 13.72%  h4: Prophet 61.13% / model 16.97%   (0.3 phút)


01:30:03 - cmdstanpy - INFO - Chain [1] start processing
01:30:09 - cmdstanpy - INFO - Chain [1] done processing


...

01:32:30 - cmdstanpy - INFO - Chain [1] start processing
01:32:36 - cmdstanpy - INFO - Chain [1] done processing


   [39/40] trạm 41: h1: Prophet 61.10% / model 15.69%  h4: Prophet 57.71% / model 21.37%   (2.8 phút)


01:32:38 - cmdstanpy - INFO - Chain [1] start processing
01:32:43 - cmdstanpy - INFO - Chain [1] done processing


   [40/40] trạm 42: h1: Prophet 59.13% / model 13.04%  h4: Prophet 54.92% / model 15.34%   (2.9 phút)

Đã chạy xong 40/40 trạm trong 2.9 phút.


,site_id,n_train,n_test_h1,wape_prophet_h1,wape_model_h1,n_test_h4,wape_prophet_h4,wape_model_h4
0,1,31374,5888,67.0709,13.2638,5457,63.1667,16.7606
1,2,30602,5868,65.2804,13.7205,5431,61.1334,16.9747
2,3,30592,5782,72.6025,14.5647,5355,67.9935,17.5040
3,4,14069,4914,43.5385,15.6804,4657,41.1793,17.5102
4,5,13872,3761,46.0920,15.5980,3540,43.8333,18.1584


## 7. Kết quả tổng hợp và Forecast Skill Score

WAPE tổng hợp bằng cách **gộp toàn bộ sai số tuyệt đối rồi chia tổng sản lượng thật**,
không lấy trung bình WAPE theo trạm. Trung bình của tỷ số không bằng tỷ số của tổng —
lấy trung bình theo trạm sẽ cho trạm nhỏ cùng trọng số với trạm lớn, làm lệch con số.

In [7]:
def bo_chi_so(y_that, y_bao):
    """Bộ chỉ số đầy đủ cho 1 cặp (thực tế, dự báo)."""
    e = y_bao - y_that
    return {
        'wape': np.abs(e).sum() / np.abs(y_that).sum() * 100,
        'rmse': np.sqrt((e ** 2).mean()),
        'mae': np.abs(e).mean(),
        'r2': 1 - (e ** 2).sum() / ((y_that - y_that.mean()) ** 2).sum(),
    }


tom_tat = {
    'nguon_hoc': 'v3_development_selected (train+val), chỉ dòng measured',
    'nguon_cham_diem': '07_final_test/prediction_audit.parquet',
    'so_site': int(len(df_ket_qua)),
    'so_site_loi': len(site_loi),
}
bang_bao_cao = []

for h in HORIZONS:
    m = mask_cham[h] & df_audit[f'prophet_h{h}'].notna()
    y_that = df_audit.loc[m, f'y_true_h{h}'].to_numpy(float)
    cs_model = bo_chi_so(y_that, df_audit.loc[m, f'y_pred_h{h}'].to_numpy(float))
    cs_prophet = bo_chi_so(y_that, df_audit.loc[m, f'prophet_h{h}'].to_numpy(float))
    ss = (1 - cs_model['wape'] / cs_prophet['wape']) * 100

    tom_tat[f'h{h}'] = {
        'n_dong': int(m.sum()),
        'wape_prophet_%': round(cs_prophet['wape'], 4),
        'wape_lightgbm_%': round(cs_model['wape'], 4),
        'skill_score_%': round(ss, 4),
    }
    for ten, cs in (('LightGBM (đủ đặc trưng thời tiết)', cs_model),
                    ('Prophet (không đặc trưng thời tiết)', cs_prophet)):
        bang_bao_cao.append({
            'horizon': f'h{h}', 'mo_hinh': ten, 'n_dong': int(m.sum()),
            'WAPE_%': round(cs['wape'], 4), 'RMSE': round(cs['rmse'], 4),
            'MAE': round(cs['mae'], 4), 'R2': round(cs['r2'], 4),
        })

    print(f'=== h{h} — {int(m.sum()):,} dòng, CÙNG tập dòng cho cả hai mô hình ===')
    print(f'   Prophet  WAPE = {cs_prophet["wape"]:7.4f}%')
    print(f'   LightGBM WAPE = {cs_model["wape"]:7.4f}%')
    print(f'   Forecast Skill Score = {ss:+7.2f}%\n')

df_bao_cao = pd.DataFrame(bang_bao_cao)
display(df_bao_cao)

=== h1 — 220,114 dòng, CÙNG tập dòng cho cả hai mô hình ===
   Prophet  WAPE = 58.3087%
   LightGBM WAPE = 17.7488%
   Forecast Skill Score =  +69.56%

=== h4 — 202,908 dòng, CÙNG tập dòng cho cả hai mô hình ===
   Prophet  WAPE = 55.0887%
   LightGBM WAPE = 21.7791%
   Forecast Skill Score =  +60.47%



,horizon,mo_hinh,n_dong,WAPE_%,RMSE,MAE,R2
0,h1,LightGBM (đủ đặc trưng thời tiết),220114,17.7488,3.2918,1.3543,0.9304
1,h1,Prophet (không đặc trưng thời tiết),220114,58.3087,8.3522,4.4493,0.5518
2,h4,LightGBM (đủ đặc trưng thời tiết),202908,21.7791,3.9489,1.7420,0.9031
3,h4,Prophet (không đặc trưng thời tiết),202908,55.0887,8.3801,4.4064,0.5637


### Nhận xét

Prophet chỉ học chu kỳ ngày/tuần từ lịch sử sản lượng, không có đặc trưng thời tiết thật
(`shortwave_radiation`, `chi_so_troi_quang`, `cloud_x_shortwave`...) nên không biết trước
những ngày mây hay thời tiết bất thường. Sai số của Prophet vì thế cao hơn hẳn LightGBM.

Chênh lệch giữa hai mô hình là **bằng chứng định lượng cho giá trị của việc đưa đặc trưng
thời tiết và bước downscale bức xạ vào mô hình**, thay vì chỉ dựa vào tính chu kỳ thời gian.

**Giới hạn diễn giải:** Skill Score đo mức cải thiện so với *một* mô hình đối chứng cụ thể,
không phải so với mọi phương pháp khả dĩ. Một baseline yếu hơn sẽ cho Skill Score cao hơn
mà không phản ánh năng lực mô hình tốt hơn. Con số này không được đọc là bằng chứng cho
tính tối ưu của mô hình.

## 8. Kiểm chứng Toàn vẹn Dữ liệu (Data QA/QC)

In [8]:
kiem_tra = []
for h in HORIZONS:
    m = mask_cham[h]
    co_prophet = m & df_audit[f'prophet_h{h}'].notna()
    kiem_tra.append({
        'horizon': f'h{h}',
        'dòng trong phạm vi': int(m.sum()),
        'dòng có dự báo Prophet': int(co_prophet.sum()),
        'thiếu (%)': round((1 - co_prophet.sum() / m.sum()) * 100, 4),
        'Prophet âm': int((df_audit.loc[co_prophet, f'prophet_h{h}'] < 0).sum()),
        'Prophet vô cực/NaN': int(
            (~np.isfinite(df_audit.loc[co_prophet, f'prophet_h{h}'])).sum()),
    })

df_qa = pd.DataFrame(kiem_tra)
display(df_qa)

assert (df_qa['Prophet âm'] == 0).all(), 'Có dự báo âm — sản lượng không thể âm'
assert (df_qa['Prophet vô cực/NaN'] == 0).all(), 'Có giá trị vô cực/NaN trong dự báo'
print('\nQA/QC đạt: không có dự báo âm, không có giá trị vô cực hay NaN.')

,horizon,dòng trong phạm vi,dòng có dự báo Prophet,thiếu (%),Prophet âm,Prophet vô cực/NaN
0,h1,220114,220114,0.0,0,0
1,h4,202908,202908,0.0,0,0



QA/QC đạt: không có dự báo âm, không có giá trị vô cực hay NaN.


### Nhận xét

Cột *thiếu (%)* cho biết tỷ lệ dòng trong phạm vi chấm điểm mà Prophet không dự báo được
— thường do trạm đó có quá ít dữ liệu lịch sử để học. Tỷ lệ này cần gần 0; nếu lớn thì
Skill Score đang được tính trên một tập con nhỏ hơn dự kiến và phải ghi rõ khi báo cáo.

## 9. Export Processed Dataset

In [9]:
# Bảng theo từng trạm
df_ket_qua.to_csv(OUTPUT_DIR / 'prophet_test_by_site.csv', index=False)

# Tóm tắt Skill Score
if site_loi:
    tom_tat['site_loi'] = site_loi
(OUTPUT_DIR / 'prophet_test_summary.json').write_text(
    json.dumps(tom_tat, ensure_ascii=False, indent=2), encoding='utf-8')

# Dự báo Prophet theo từng dòng — dashboard đọc file này để vẽ đường đối chứng
cot_prophet = [c for c in df_audit.columns if c.startswith('prophet_h')]
df_audit[['site_id', 'timestamp', *cot_prophet]].to_parquet(
    OUTPUT_DIR / 'prophet_test_predictions.parquet', index=False)

# Bảng báo cáo gọn để dán vào LaTeX
df_bao_cao.to_csv(OUTPUT_DIR / 'prophet_bang_bao_cao.csv', index=False)

print('Đã ghi 4 tệp:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name:38s} {p.stat().st_size / 1024:8.1f} KB')

Đã ghi 4 tệp:
  prophet_bang_bao_cao.csv                    0.4 KB
  prophet_test_by_site.csv                    2.1 KB
  prophet_test_predictions.parquet         4787.3 KB
  prophet_test_summary.json                   0.4 KB
